In [1]:
"""
CoRAG -- Chain-of-Retrieval Augmented Generation
=================================================
Reference   : Wang et al., "Chain-of-Retrieval Augmented Generation"
              arXiv: 2501.14342v3  (NeurIPS 2025, Microsoft Research)
              Code: https://github.com/microsoft/LMOps/tree/main/corag

Architecture: FIVE configurations covering the full CoRAG design space:
              (1) Baseline Single-Step RAG        -- one retrieve-then-generate pass
              (2) CoRAG Greedy Chain              -- sequential sub-query chain,
                                                    greedy decoding, fixed depth L
              (3) CoRAG Best-of-N Sampling        -- sample N chains, select by
                                                    minimum "No relevant info" penalty
              (4) CoRAG BFS Tree Search           -- breadth-first search with
                                                    parallel rollouts at each step,
                                                    select lowest-penalty path
              (5) CoRAG Iterative Self-Correct    -- greedy chain with self-
                  [BEST inference-time strategy]    correction: detects retrieval
                                                    failures and reformulates
                                                    queries before continuing chain

=============================================================================
CORE PROBLEM: WHY SINGLE-STEP RETRIEVAL FAILS ON MULTI-HOP QUERIES
=============================================================================
Conventional RAG: one retrieval step with the original query Q, producing
a fixed context C = Retrieve(Q), then generating answer = LLM(Q, C).

The core failure mode is that complex queries require information from
multiple, independently-stored documents that are ONLY linked through
an intermediate reasoning step. Consider:

    Q: "Where did the star of Dark Hazard go to college?"

    Retrieval with Q returns:   Documents about "Dark Hazard" (the film).
    What you need first:        The star's name ("Edward G. Robinson").
    What you need second:       Where Edward G. Robinson studied.

    A bi-encoder embeds "star of Dark Hazard college" into one fixed-size
    vector. This vector must simultaneously encode film identity, actor
    identity, and academic institution -- three independent semantic fields.
    The embedding compression loses at least one of these fields, resulting
    in retrieval that typically returns only film-related documents.

    The 10+ EM-point gap between CoRAG and standard RAG on multi-hop QA
    benchmarks (Wang et al., 2025) is entirely explained by this mismatch
    between query complexity and single-step retrieval capacity.

=============================================================================
CORAG CHAIN-OF-RETRIEVAL ARCHITECTURE
=============================================================================
CoRAG models retrieval as a sequential decision process:

    State s_t = (Q, Q_1:t, A_1:t)
        where Q           = original query
              Q_1:t       = sub-queries issued so far (the "retrieval chain")
              A_1:t       = sub-answers received so far

    Action a_t = next sub-query Q_t+1 = LLM(s_t)
        The LLM reads the ENTIRE current state (original query + all previous
        sub-query/sub-answer pairs) and generates the NEXT targeted sub-query.
        This sub-query is grounded in the evolving knowledge state, NOT the
        raw original question.

    Observation o_t = sub-answer A_t = LLM(Q_t, Retrieve(Q_t, top_k))
        Retrieved documents are fed to the LLM to answer the sub-query.
        If no relevant document is found: A_t = "No relevant information found."
        This sentinel response TRIGGERS query reformulation in the next step.

    Termination: when A_t contains the final answer OR chain length reaches L.

TRAINING (production only, not re-implemented here, reference only):
    Rejection sampling builds chains:
        Sample M chains per query using Llama-3.1-8B-Instruct.
        Score each chain: log P(A | Q, Q_1:L, A_1:L) -- log-likelihood of
        the correct answer given the full chain.
        Select the highest-log-likelihood chain.
    Fine-tune Llama-3.1-8B-Instruct with three losses:
        L_sub_query  = -log P(Q_i | Q, Q_<i, A_<i)
        L_sub_answer = -log P(A_i | Q_i, D^(i)_1:k)
        L_final_ans  = -log P(A | Q, Q_1:L, A_1:L, D_1:k)
    Result: a model that has internalized the retrieve-step-by-step behavior
    through supervised imitation of high-quality rejection-sampled chains.

THIS PIPELINE: We implement the INFERENCE-TIME behaviors of CoRAG using
    Azure OpenAI (GPT-4o) as the LLM and BGE-large + FAISS as the retriever.
    The five configurations correspond to CoRAG's three test-time strategies
    plus ablations of the chain mechanism itself.

=============================================================================
TEST-TIME SCALING: THE PARETO FRONTIER
=============================================================================
Wang et al. (2025) find that across decoding strategies, the Pareto frontier
approximately adheres to a LOG-LINEAR relationship between total token
consumption and model performance. This is the RAG analog of the compute-
performance scaling laws in pre-training and reasoning.

    Greedy chain (L=2):  low tokens, modest improvement over single-step
    Greedy chain (L=5):  medium tokens, substantial improvement
    Best-of-N (N=4):     medium tokens, better than greedy of same compute
    Tree search:         high tokens, best performance on hardest queries
    Iterative correct:   adaptive tokens, best cost-efficiency in practice

The implication for production systems: when a query is detected as easy
(simple factoid, high retrieval recall), skip iterative retrieval.
When detected as hard (multi-hop, low retrieval confidence), allocate extra
steps. CoRAG exhibits this dynamically -- simple queries terminate early,
complex queries automatically extend their chain length.

=============================================================================
PENALTY SCORE: "NO RELEVANT INFORMATION FOUND"
=============================================================================
Best-of-N and tree search require a proxy for chain quality AT TEST TIME
(ground-truth answer not available). CoRAG uses:

    penalty(chain) = -log P("No relevant information found" | Q, chain)

Rationale: a good retrieval chain leaves the model well-positioned to answer
the original query. If the model is inclined to say "No relevant info found"
after reading the chain, the chain was poor. The LOWEST penalty score
(the model is LEAST likely to say "no info") is selected.

In this production implementation: we use the LLM to rate chain quality
on a 1-10 scale (chain assessment prompt) as a practical Azure OpenAI-
compatible proxy. Logit-level penalty requires an open-weights model.


"""


'\nCoRAG -- Chain-of-Retrieval Augmented Generation\n=================================================\nReference   : Wang et al., "Chain-of-Retrieval Augmented Generation"\n              arXiv: 2501.14342v3  (NeurIPS 2025, Microsoft Research)\n              Code: https://github.com/microsoft/LMOps/tree/main/corag\n\nArchitecture: FIVE configurations covering the full CoRAG design space:\n              (1) Baseline Single-Step RAG        -- one retrieve-then-generate pass\n              (2) CoRAG Greedy Chain              -- sequential sub-query chain,\n                                                    greedy decoding, fixed depth L\n              (3) CoRAG Best-of-N Sampling        -- sample N chains, select by\n                                                    minimum "No relevant info" penalty\n              (4) CoRAG BFS Tree Search           -- breadth-first search with\n                                                    parallel rollouts at each step,\n                      

In [2]:

# ─────────────────────────────────────────────────────────────────────
# IMPORTS
# ─────────────────────────────────────────────────────────────────────
from __future__ import annotations

import os
import re
import ast
import sys
import json
import math
import time
import hashlib
import textwrap
import warnings
import operator
import datetime
import traceback
from enum import Enum
from copy import deepcopy
from pathlib import Path
from typing import (
    Any, Callable, Dict, List, Optional,
    Tuple, Union, NamedTuple
)
from dataclasses import dataclass, field
import logging
import urllib.request
import urllib.error

warnings.filterwarnings("ignore")


In [3]:
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import Chroma, FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import (
    EnsembleRetriever,
    ContextualCompressionRetriever,
)
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate,SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain_core.output_parsers import BaseOutputParser, StrOutputParser
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_core.runnables import RunnablePassthrough ,RunnableLambda
from sentence_transformers import CrossEncoder
from transformers import T5ForConditionalGeneration, T5Tokenizer

W0525 22:10:08.875000 3900 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [4]:

# ===========================================================================
# LOGGING
# ===========================================================================
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)],
)
logger = logging.getLogger("corag")


In [5]:


# ===========================================================================
# SECTION 1 -- CONFIGURATION
# ===========================================================================

class CoRAGConfig:
    """
    Centralized configuration for the CoRAG pipeline.

    CHAIN_LENGTH (L=4):
        Maximum number of iterative retrieval steps per query.
        Wang et al. (2025) use L in {2, 3, 5} in ablations.
        At L=4: three sub-questions bridge an average 3-hop reasoning chain
        with one additional step for verification or edge cases.
        Cost: each step = 2 LLM calls (sub-query generation + sub-answer) +
              1 retrieval call. At L=4: up to 9 LLM calls total.
        Early termination fires when A_i contains the final answer,
        so actual average chain length is typically 2-3 for most queries.

    TOP_K_RETRIEVAL (5):
        Number of documents retrieved per sub-query.
        The sub-query is targeted (e.g., "Where did Edward G. Robinson study?")
        and more precise than the original query, so top-k=5 provides sufficient
        coverage for answering the sub-question without flooding the context.
        Compare to single-step RAG which typically uses top-k=4-10 for the
        full query. Here k=5 per step gives similar total coverage with better
        precision because each step's query is semantically focused.

    BEST_OF_N (N=4):
        Number of chains sampled in Config 3 (Best-of-N Sampling).
        Wang et al. (2025) test N in {1, 2, 4, 8, 16}.
        At N=4: significant quality improvement over greedy (N=1) with a
        manageable 4x LLM cost multiplier. N=8 and above have diminishing
        returns unless using the fine-tuned model (which has higher variance).

    BFS_BEAM_WIDTH (2):
        Number of candidate sub-queries expanded at each BFS tree step.
        Wang et al. (2025) use beam_width x rollouts to control tree search.
        beam_width=2 means at each step we consider 2 candidate next sub-queries
        (generated with temperature>0) and score each with rollout_count rollouts.

    BFS_ROLLOUT_COUNT (2):
        Number of rollouts per expanded state in tree search.
        A rollout completes the chain from the current state to the end
        (up to remaining steps) and computes the average penalty score.
        Each rollout is a greedy completion.

    ANSWER_TEMPERATURE (0.0):
        Deterministic answer generation. The final answer step uses greedy
        decoding to ensure reproducibility.

    CHAIN_TEMPERATURE (0.7):
        Stochastic sub-query generation for Best-of-N and tree search.
        At 0.7: meaningful diversity in sub-query formulation without
        collapse to nonsense (which happens at T > 1.0 for factual tasks).
        For greedy chain (Config 2): temperature is irrelevant (greedy=argmax).

    RETRIEVAL_FAILURE_SENTINEL:
        The exact string that triggers query reformulation in Config 5.
        When a sub-answer contains this sentinel, the next sub-query is
        explicitly a REFORMULATION of the failed sub-query, not a new sub-question.
        This is the self-correction mechanism of Config 5.

    MAX_REFORMULATION_ATTEMPTS (2):
        Maximum consecutive reformulation attempts before advancing to the
        next reasoning step. Prevents infinite loops on irretrievable sub-questions.
    """

    # --- Dataset -----------------------------------------------------------
    PDF_SOURCES: List[Tuple[str, str]] = [
        ("attention_is_all_you_need",    "https://arxiv.org/pdf/1706.03762.pdf"),
        ("bert_pretraining_transformers", "https://arxiv.org/pdf/1810.04805.pdf"),
        ("rag_knowledge_intensive_nlp",  "https://arxiv.org/pdf/2005.11401.pdf"),
    ]
    PDF_DOWNLOAD_DIR: str = "./pdfs"
    FAISS_DIR:        str = "./faiss_corag_index"

    # --- BGE Embeddings ----------------------------------------------------
    BIENCODER_MODEL:             str = "BAAI/bge-large-en-v1.5"
    BIENCODER_DEVICE:            str = "cpu"
    BIENCODER_QUERY_INSTRUCTION: str = (
        "Represent this sentence for searching relevant passages: "
    )

    # --- Chunking ----------------------------------------------------------
    CHUNK_SIZE:    int = 450
    CHUNK_OVERLAP: int = 60

    # --- CoRAG Chain Parameters --------------------------------------------
    CHAIN_LENGTH:         int = 4     # max retrieval steps L
    TOP_K_RETRIEVAL:      int = 5     # docs retrieved per sub-query
    FLAT_TOP_K:           int = 4     # baseline single-step top-k

    # --- Best-of-N Parameters ----------------------------------------------
    BEST_OF_N:         int   = 4     # number of chains to sample
    CHAIN_TEMPERATURE: float = 0.7   # sampling temperature for chain steps

    # --- Tree Search Parameters --------------------------------------------
    BFS_BEAM_WIDTH:    int = 2   # candidate sub-queries expanded per step
    BFS_ROLLOUT_COUNT: int = 2   # rollouts per expanded state

    # --- LLM ---------------------------------------------------------------
    ANSWER_TEMPERATURE:      float = 0.0
    OLLAMA_BASE_URL: str = os.environ.get("OLLAMA_BASE_URL", "http://localhost:11434")
    OLLAMA_MODEL:    str = os.environ.get("OLLAMA_MODEL",    "qwen2.5-coder:7b")
    LLM_MAX_TOKENS: int            = 512

    # --- Self-Correction ---------------------------------------------------
    RETRIEVAL_FAILURE_SENTINEL:   str = "no relevant information"
    MAX_REFORMULATION_ATTEMPTS:   int = 2

    # --- Context -----------------------------------------------------------
    MAX_CONTEXT_CHARS: int = 3200   # ~800 tokens for sub-answer generation

    # ==========================================================================
    # PROMPTS
    # ==========================================================================

    # Used by all configs with chain retrieval to generate the next sub-query.
    # The model reads the ENTIRE state: original question + prior chain.
    P_NEXT_SUBQUERY: str = """You are a step-by-step research assistant using iterative retrieval to answer a complex question.

Original question: {question}

Prior retrieval steps (sub-queries and sub-answers so far):
{chain_history}

Based on what you have found so far, generate ONE focused follow-up question that will help you answer the original question.
The follow-up question must:
- Target a SPECIFIC piece of information still needed.
- Be SHORT (5-12 words).
- Be a standalone question retrievable from a document corpus.
- NOT repeat a prior sub-query.

If you have all information needed to answer the original question, respond only with: DONE

Follow-up question:"""

    # Used in Config 5 (self-correction) when retrieval fails.
    P_REFORMULATE_SUBQUERY: str = """The following search query returned no useful results:
Original query: {failed_query}
Context so far: {chain_history}

Reformulate the query into a different phrasing that might return better results.
Keep it SHORT (5-12 words). Output ONLY the reformulated query, nothing else.

Reformulated query:"""

    # Generate a sub-answer from retrieved documents for a sub-query.
    P_SUB_ANSWER: str = """Answer the following question using ONLY the provided context.
If the context does not contain the answer, respond ONLY with:
"No relevant information found."

Context:
{context}

Question: {subquery}

Answer (1-2 sentences, be specific):"""

    # Generate the FINAL answer from the complete chain.
    P_FINAL_ANSWER: str = """You are a precise research assistant.
Using the retrieval chain below (sub-questions and their answers),
synthesize a COMPLETE, PRECISE final answer to the original question.
Do not include phrases like "based on the chain" -- give the direct answer.
If the chain does not provide enough information, say:
"The provided documents do not contain sufficient information to answer this question."

Original question: {question}

Retrieval chain:
{chain_history}

Final answer:"""

    # Baseline single-step answer prompt (Config 1).
    P_BASELINE_ANSWER: str = """You are a precise technical research assistant.
Answer the question using ONLY the information in the context below.
If the answer is not present, respond:
"The provided documents do not contain sufficient information."

Context:
{context}

Question: {question}

Precise answer:"""

    # Chain quality assessment proxy (used instead of logit-level penalty).
    # Returns a score 1-10; HIGHER score = BETTER chain (less failure signal).
    P_CHAIN_QUALITY: str = """Rate the following retrieval chain's quality for answering the given question.
A HIGH score means the chain progressively builds toward the answer.
A LOW score means the chain fails to retrieve relevant information or goes off-track.

Original question: {question}

Retrieval chain:
{chain_history}

Rate the chain quality from 1 (completely unhelpful) to 10 (perfectly targeted).
Respond with ONLY a single integer (1-10), nothing else.

Score:"""



In [6]:


# ===========================================================================
# SECTION 2 -- DATA STRUCTURES
# ===========================================================================

@dataclass
class RetrievalStep:
    """
    One step in a CoRAG retrieval chain.

    sub_query:    The targeted sub-question generated by the LLM at this step.
    retrieved:    Top-k documents retrieved for sub_query.
    sub_answer:   LLM's answer to sub_query given retrieved documents.
    is_failure:   True if sub_answer contains the failure sentinel
                  ("no relevant information found").
    is_terminal:  True if sub_answer contains the final answer to the
                  original question (chain terminates here).
    reformulated: True if this step is a reformulation of the previous
                  failed sub_query (Config 5 self-correction).
    """
    step_num:       int
    sub_query:      str
    retrieved:      List[Document] = field(default_factory=list)
    sub_answer:     str = ""
    is_failure:     bool = False
    is_terminal:    bool = False
    reformulated:   bool = False
    retrieval_ms:   float = 0.0
    generation_ms:  float = 0.0

    def to_text(self) -> str:
        """Format this step for inclusion in chain history context."""
        prefix = "[REFORMULATED] " if self.reformulated else ""
        failure_tag = " [RETRIEVAL FAILED]" if self.is_failure else ""
        return (f"Step {self.step_num}: {prefix}Sub-query: {self.sub_query}\n"
                f"Sub-answer: {self.sub_answer}{failure_tag}")


@dataclass
class RetrievalChain:
    """
    A complete retrieval chain for one query.
    Contains the sequence of RetrievalSteps from step 1 through termination.
    """
    question:       str
    steps:          List[RetrievalStep] = field(default_factory=list)
    final_answer:   str = ""
    quality_score:  float = 0.0   # proxy for log P(final_answer | chain)
    total_tokens:   int   = 0     # approximate (sub-queries + sub-answers)
    total_llm_calls:int   = 0

    def chain_history_text(self) -> str:
        if not self.steps:
            return "No retrieval steps taken yet."
        return "\n".join(step.to_text() for step in self.steps)

    def n_steps(self) -> int:
        return len(self.steps)

    def n_failures(self) -> int:
        return sum(1 for s in self.steps if s.is_failure)


@dataclass
class CoRAGTrace:
    """
    Execution trace for one CoRAG query across all five configurations.
    """
    question:       str
    strategy:       str
    chains:         List[RetrievalChain] = field(default_factory=list)
    selected_chain: Optional[RetrievalChain] = None
    final_answer:   str  = ""
    retrieval_ms:   float = 0.0
    generation_ms:  float = 0.0
    total_ms:       float = 0.0
    llm_calls:      int  = 0

    def print_trace(self) -> None:
        print(f"\n{'='*74}")
        print(f"TRACE: {self.strategy}")
        print(f"Query: {self.question[:74]}")
        print(f"{'='*74}")
        sc = self.selected_chain
        if sc:
            print(f"  Chain steps:   {sc.n_steps()} | "
                  f"Retrieval failures: {sc.n_failures()} | "
                  f"Quality score: {sc.quality_score:.2f}")
            print(f"\n  CHAIN HISTORY:")
            for step in sc.steps:
                fail_tag = " [FAIL]" if step.is_failure else ""
                reform_tag = " [REFORM]" if step.reformulated else ""
                print(f"  {step.step_num}{fail_tag}{reform_tag}. "
                      f"Q: {step.sub_query[:60]}")
                print(f"     A: {step.sub_answer[:100]}")
        print(f"\n  LLM calls: {self.llm_calls}")
        print(f"  Latency: retrieval={self.retrieval_ms:.0f}ms  "
              f"gen={self.generation_ms:.0f}ms  total={self.total_ms:.0f}ms")
        print(f"\n  FINAL ANSWER:\n  {self.final_answer[:450]}")
        print("=" * 74 + "\n")



In [7]:


# ===========================================================================
# SECTION 3 -- INFRASTRUCTURE
# ===========================================================================

def download_pdfs(config: CoRAGConfig) -> List[Path]:
    dl_dir = Path(config.PDF_DOWNLOAD_DIR)
    dl_dir.mkdir(parents=True, exist_ok=True)
    paths: List[Path] = []
    for name, url in config.PDF_SOURCES:
        dest = dl_dir / f"{name}.pdf"
        if dest.exists() and dest.stat().st_size > 10_000:
            logger.info("Cached: %s", dest.name)
            paths.append(dest)
            continue
        logger.info("Downloading: %s", url)
        try:
            req = urllib.request.Request(
                url, headers={"User-Agent": "Mozilla/5.0 (CoRAG/1.0)"}
            )
            with urllib.request.urlopen(req, timeout=60) as resp:
                data = resp.read()
            dest.write_bytes(data)
            paths.append(dest)
            time.sleep(1.0)
        except Exception as exc:
            raise RuntimeError(f"Cannot download '{url}': {exc}") from exc
    return paths


def load_and_chunk(pdf_paths: List[Path], config: CoRAGConfig) -> List[Document]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=config.CHUNK_SIZE, chunk_overlap=config.CHUNK_OVERLAP,
        separators=["\n\n", "\n", ". ", " ", ""],
        add_start_index=True,
    )
    all_chunks: List[Document] = []
    for pdf_path in pdf_paths:
        paper_name = pdf_path.stem.replace("_", " ").title()
        pages = PyPDFLoader(str(pdf_path)).load()
        for page in pages:
            page.metadata["paper_name"] = paper_name
            page.metadata["source"]     = pdf_path.name
        chunks = splitter.split_documents(pages)
        logger.info("  %s -> %d chunks", pdf_path.name, len(chunks))
        all_chunks.extend(chunks)
    logger.info("Total chunks: %d", len(all_chunks))
    return all_chunks


def build_bge_embeddings(config: CoRAGConfig) -> HuggingFaceEmbeddings:
    logger.info("Loading BGE: %s", config.BIENCODER_MODEL)
    return HuggingFaceEmbeddings(
        model_name=config.BIENCODER_MODEL,
        model_kwargs={"device": config.BIENCODER_DEVICE},
        encode_kwargs={"normalize_embeddings": True},
    )


def build_faiss(chunks: List[Document], embeddings: HuggingFaceEmbeddings,
                config: CoRAGConfig) -> FAISS:
    faiss_file = Path(config.FAISS_DIR) / "index.faiss"
    if faiss_file.exists():
        try:
            vs = FAISS.load_local(config.FAISS_DIR, embeddings,
                                  allow_dangerous_deserialization=True)
            logger.info("FAISS loaded. Vectors: %d", vs.index.ntotal)
            return vs
        except Exception:
            pass
    logger.info("Building FAISS from %d chunks ...", len(chunks))
    vs = FAISS.from_documents(chunks, embeddings)
    Path(config.FAISS_DIR).mkdir(parents=True, exist_ok=True)
    vs.save_local(config.FAISS_DIR)
    return vs


def build_ollama_llm(config: CoRAGConfig,
                    temperature: float = 0.0) -> ChatOllama:
    """Connect to local Ollama and return a ChatOllama instance."""
    import urllib.request, urllib.error
    base_url = getattr(config, 'OLLAMA_BASE_URL', 'http://localhost:11434')
    model    = getattr(config, 'OLLAMA_MODEL',    'qwen2.5-coder:7b')
    try:
        urllib.request.urlopen(base_url, timeout=3)
    except Exception as exc:
        raise RuntimeError(
            f"Ollama not reachable at {base_url}. Start Ollama and run: "
            f"ollama pull qwen2.5-coder:7b"
        ) from exc
    return ChatOllama(
        base_url=base_url,
        model=model,
        temperature=getattr(config, 'LLM_TEMPERATURE', temperature),
        num_predict=getattr(config, 'LLM_MAX_TOKENS', 512),
    )


def llm_call(prompt: str, llm: ChatOllama) -> str:
    """Single LLM call returning raw response string."""
    return llm.invoke([HumanMessage(content=prompt)]).content.strip()


In [8]:

# ===========================================================================
# SECTION 4 -- CORE CHAIN BUILDING UTILITIES
# ===========================================================================

def retrieve_for_subquery(
    subquery: str, vs: FAISS, k: int,
) -> Tuple[List[Document], float]:
    """
    Retrieve top-k documents for a sub-query using FAISS cosine similarity.

    Returns:
        (documents, retrieval_latency_ms)
    """
    t0 = time.perf_counter()
    docs = vs.similarity_search(subquery, k=k)
    return docs, (time.perf_counter() - t0) * 1000


def build_sub_context(docs: List[Document], max_chars: int) -> str:
    """
    Build context string from retrieved documents for sub-answer generation.
    Includes paper name, page, and chunk content. Truncated to max_chars.
    """
    parts = []
    total = 0
    for doc in docs:
        paper = doc.metadata.get("paper_name", "?")[:25]
        page  = doc.metadata.get("page", "?")
        part  = f"[{paper} p{page}] {doc.page_content.strip()}"
        if total + len(part) > max_chars:
            remaining = max_chars - total
            if remaining > 80:
                parts.append(part[:remaining])
            break
        parts.append(part)
        total += len(part)
    return "\n\n---\n\n".join(parts)


def generate_next_subquery(
    question: str, chain: RetrievalChain,
    llm: ChatOllama, config: CoRAGConfig,
    temperature: float = 0.0,
) -> str:
    """
    Generate the next sub-query given the original question and chain history.

    This is the CORE ACTION of the CoRAG decision process:
        Q_{t+1} = LLM(Q, Q_{1:t}, A_{1:t})

    The LLM reads the entire current state and decides what to retrieve next.
    The model can generate "DONE" to signal early termination.

    Args:
        question    : Original user question.
        chain       : Current chain state (includes all prior steps).
        llm         : ChatOllama.
        config      : CoRAGConfig.
        temperature : Sampling temperature (0.0=greedy, 0.7=diverse).

    Returns:
        Next sub-query string, or "DONE" to terminate.
    """
    history_text = chain.chain_history_text() if chain.steps else "None yet."
    prompt = config.P_NEXT_SUBQUERY.format(
        question=question,
        chain_history=history_text,
    )
    # Use temperature-controlled LLM by rebuilding with correct temp
    # NOTE: for greedy (temp=0.0) we use the pre-built llm directly.
    # For sampling (temp>0.0): build a new instance at the right temperature.
    if temperature > 0.0:
        sampling_llm = build_ollama_llm(config, temperature=temperature)
        return sampling_llm.invoke([HumanMessage(content=prompt)]).content.strip()
    return llm_call(prompt, llm)


def generate_sub_answer(
    subquery: str, docs: List[Document],
    llm: ChatOllama, config: CoRAGConfig,
) -> Tuple[str, bool, bool]:
    """
    Generate a sub-answer from retrieved documents for a given sub-query.

    Detects:
        is_failure:  sub-answer matches the failure sentinel -- no useful docs.
        is_terminal: sub-answer appears to directly answer the original question
                     (detected by checking length and absence of failure sentinel).

    Args:
        subquery : The targeted sub-question.
        docs     : Retrieved documents for this sub-query.
        llm      : ChatOllama.
        config   : CoRAGConfig.

    Returns:
        (sub_answer_text, is_failure, is_terminal)
    """
    context = build_sub_context(docs, config.MAX_CONTEXT_CHARS)
    prompt  = config.P_SUB_ANSWER.format(
        context=context or "No documents retrieved.",
        subquery=subquery,
    )
    answer = llm_call(prompt, llm)

    is_failure = (
        config.RETRIEVAL_FAILURE_SENTINEL in answer.lower()
        or len(answer.strip()) < 10
    )
    # Terminal heuristic: answer is short and factual (a name, date, place, number)
    # This is a prompt-based proxy for the fine-tuned model's native termination.
    is_terminal = (not is_failure and len(answer.split()) <= 20)

    return answer, is_failure, is_terminal


def generate_reformulated_subquery(
    failed_query: str, chain: RetrievalChain,
    llm: ChatOllama, config: CoRAGConfig,
) -> str:
    """
    Reformulate a failed sub-query into a different phrasing.

    This is the SELF-CORRECTION mechanism of Config 5. When the retriever
    returns no useful documents for a sub-query, the model reformulates the
    sub-query into a different phrasing before retrying. This corresponds to
    the "dynamic query reformulation" capability described in Wang et al. (2025)
    where CoRAG "self-corrects by verifying information through additional
    retrieval steps."

    Args:
        failed_query : The sub-query that returned no useful results.
        chain        : Current chain state for context.
        llm          : ChatOllama.
        config       : CoRAGConfig.

    Returns:
        Reformulated sub-query string.
    """
    history_text = chain.chain_history_text()
    prompt = config.P_REFORMULATE_SUBQUERY.format(
        failed_query=failed_query,
        chain_history=history_text,
    )
    return llm_call(prompt, llm)


def generate_final_answer(
    question: str, chain: RetrievalChain,
    llm: ChatOllama, config: CoRAGConfig,
) -> str:
    """
    Generate the final answer from the complete retrieval chain.

    The model reads the entire chain (all sub-query/sub-answer pairs) and
    synthesizes the final answer. This is the ANSWER GENERATION step that
    consumes the chain built during iterative retrieval.

    Training objective counterpart: L_final_answer = -log P(A | Q, Q_1:L, A_1:L, D_1:k)

    Args:
        question : Original user question.
        chain    : Complete retrieval chain.
        llm      : ChatOllama.
        config   : CoRAGConfig.

    Returns:
        Final answer string.
    """
    prompt = config.P_FINAL_ANSWER.format(
        question=question,
        chain_history=chain.chain_history_text(),
    )
    return llm_call(prompt, llm)


def score_chain_quality(
    question: str, chain: RetrievalChain,
    llm: ChatOllama, config: CoRAGConfig,
) -> float:
    """
    Score a retrieval chain's quality as a proxy for -log P(failure | Q, chain).

    In the CoRAG paper, chain quality is measured by:
        penalty(chain) = log P("No relevant information found" | Q, chain)
    This requires access to raw logits from an open-weights model.

    As an Azure OpenAI-compatible proxy, we prompt the LLM to score the chain
    1-10. HIGHER is better (lower failure probability). The score is normalized
    to [0, 1] for use in Best-of-N selection and tree search.

    For production with open-weights models (Llama-3.1-8B): replace this
    function with the actual log-likelihood computation:
        from transformers import AutoModelForCausalLM, AutoTokenizer
        inputs = tokenizer(prompt + "No relevant information found.", ...)
        logits = model(**inputs).logits
        penalty = -log_proba_of_sentinel_tokens(logits)

    Args:
        question : Original question.
        chain    : RetrievalChain to score.
        llm      : ChatOllama.
        config   : CoRAGConfig.

    Returns:
        Quality score in [0, 1]. Higher = better chain.
    """
    prompt = config.P_CHAIN_QUALITY.format(
        question=question,
        chain_history=chain.chain_history_text(),
    )
    raw = llm_call(prompt, llm)
    try:
        score = int(raw.strip().split()[0])
        return max(1, min(10, score)) / 10.0
    except (ValueError, IndexError):
        return 0.5   # neutral fallback



In [9]:

# ===========================================================================
# SECTION 5 -- FIVE CORAG CONFIGURATIONS
# ===========================================================================

# ---------------------------------------------------------------------------
# CONFIG 1: Baseline -- single-step RAG
# ---------------------------------------------------------------------------

def run_config1_baseline(
    question: str, vs: FAISS,
    llm: ChatOllama, config: CoRAGConfig,
) -> CoRAGTrace:
    """
    Configuration 1: Standard single-step RAG.

    Retrieves FLAT_TOP_K documents using the original query and generates
    the answer in a single generation step. Control condition: establishes
    the performance floor that CoRAG Configs 2-5 improve upon.

    On multi-hop queries: this config typically fails because:
        (a) The original query is too complex for the bi-encoder embedding.
        (b) All retrieved documents address only the first reasoning hop.
        (c) The LLM cannot bridge the gap to the final answer without
            intermediate retrieved facts.

    LLM calls: 1.
    """
    trace = CoRAGTrace(question=question, strategy="Config1_Baseline_SingleStep")
    t0 = time.perf_counter()

    docs, ret_ms = retrieve_for_subquery(question, vs, config.FLAT_TOP_K)
    trace.retrieval_ms = ret_ms

    context = build_sub_context(docs, config.MAX_CONTEXT_CHARS)
    prompt  = ChatPromptTemplate.from_template(config.P_BASELINE_ANSWER)
    t_gen   = time.perf_counter()
    answer  = (prompt | llm | StrOutputParser()).invoke(
        {"context": context, "question": question}
    )
    trace.generation_ms = (time.perf_counter() - t_gen) * 1000
    trace.final_answer  = answer
    trace.llm_calls     = 1

    fake_chain = RetrievalChain(question=question)
    fake_chain.final_answer = answer
    trace.selected_chain = fake_chain
    trace.total_ms = (time.perf_counter() - t0) * 1000
    return trace


In [10]:

# ---------------------------------------------------------------------------
# CONFIG 2: CoRAG Greedy Chain (sequential, deterministic)
# ---------------------------------------------------------------------------

def _build_greedy_chain(
    question: str, vs: FAISS,
    llm: ChatOllama, config: CoRAGConfig,
    max_steps: int = None,
    temperature: float = 0.0,
) -> RetrievalChain:
    """
    Build ONE retrieval chain using greedy (or temperature-controlled) decoding.

    Each step:
        1. Generate next sub-query Q_t = LLM(Q, Q_{1:t-1}, A_{1:t-1}).
           If LLM outputs "DONE": terminate chain early.
        2. Retrieve top-k documents D^(t)_{1:k} = Retrieve(Q_t).
        3. Generate sub-answer A_t = LLM(Q_t, D^(t)_{1:k}).
           If A_t is a failure sentinel: flag step as retrieval failure.
           If A_t is a terminal (factual short answer): optionally terminate.

    This is the CORE loop of CoRAG. The fine-tuned model in the paper has
    internalized this loop through training on rejection-sampled chains;
    here we implement the inference-time behavior using GPT-4o as the LLM.

    Args:
        question    : User query.
        vs          : FAISS vectorstore for retrieval.
        llm         : ChatOllama.
        config      : CoRAGConfig.
        max_steps   : Override config.CHAIN_LENGTH if provided.
        temperature : Sub-query generation temperature.

    Returns:
        Built RetrievalChain.
    """
    L = max_steps if max_steps is not None else config.CHAIN_LENGTH
    chain = RetrievalChain(question=question)
    total_llm_calls = 0

    for step_num in range(1, L + 1):
        # --- Generate next sub-query (1 LLM call) -------------------------
        t_sq = time.perf_counter()
        next_q = generate_next_subquery(
            question, chain, llm, config, temperature=temperature
        )
        total_llm_calls += 1
        sq_ms = (time.perf_counter() - t_sq) * 1000

        # Early termination signal from LLM
        if "DONE" in next_q.upper() and len(next_q) < 10:
            logger.info("  Chain: step %d -- LLM signaled DONE", step_num)
            break

        # Truncate runaway sub-queries
        next_q = next_q[:200]

        # --- Retrieve documents -------------------------------------------
        docs, ret_ms = retrieve_for_subquery(next_q, vs, config.TOP_K_RETRIEVAL)

        # --- Generate sub-answer (1 LLM call) -----------------------------
        t_sa = time.perf_counter()
        sub_ans, is_fail, is_terminal = generate_sub_answer(
            next_q, docs, llm, config
        )
        total_llm_calls += 1
        sa_ms = (time.perf_counter() - t_sa) * 1000

        step = RetrievalStep(
            step_num=step_num, sub_query=next_q, retrieved=docs,
            sub_answer=sub_ans, is_failure=is_fail, is_terminal=is_terminal,
            retrieval_ms=ret_ms, generation_ms=sq_ms + sa_ms,
        )
        chain.steps.append(step)

        if is_fail:
            logger.info(
                "  Chain step %d: RETRIEVAL FAILED for '%s ...'", step_num, next_q[:40]
            )
        else:
            logger.info(
                "  Chain step %d: '%s' -> '%s ...'",
                step_num, next_q[:40], sub_ans[:40],
            )

        # Early termination on definitive sub-answer
        if is_terminal and step_num >= 2:
            logger.info("  Chain: terminal answer reached at step %d", step_num)
            break

    chain.total_llm_calls = total_llm_calls
    return chain


def run_config2_greedy_chain(
    question: str, vs: FAISS,
    llm: ChatOllama, config: CoRAGConfig,
) -> CoRAGTrace:
    """
    Configuration 2: CoRAG Greedy Chain.

    Builds a single retrieval chain using greedy (temperature=0) sub-query
    generation. This is the most compute-efficient CoRAG configuration:
        LLM calls: 2 per step (sub-query + sub-answer) + 1 (final answer)
        For L=4: up to 9 LLM calls total.

    Key property: DETERMINISTIC. The same query always produces the same chain.
    This is the baseline iterative config from which Configs 3/4 add variance.

    Performance vs Config 1: more than 10 EM points on multi-hop QA datasets
    (Wang et al., 2025) because the chain decomposes the query into solvable
    single-hop sub-questions and bridges the reasoning gap.
    """
    trace = CoRAGTrace(question=question, strategy="Config2_CoRAG_GreedyChain")
    t0 = time.perf_counter()

    chain = _build_greedy_chain(question, vs, llm, config)

    t_gen   = time.perf_counter()
    chain.final_answer = generate_final_answer(question, chain, llm, config)
    trace.generation_ms = (time.perf_counter() - t_gen) * 1000

    chain.quality_score = 1.0  # greedy chain: no competing chains to score
    trace.selected_chain = chain
    trace.chains         = [chain]
    trace.final_answer   = chain.final_answer
    trace.retrieval_ms   = sum(s.retrieval_ms for s in chain.steps)
    trace.llm_calls      = chain.total_llm_calls + 1  # +1 for final answer
    trace.total_ms       = (time.perf_counter() - t0) * 1000
    return trace



In [11]:


# ---------------------------------------------------------------------------
# CONFIG 3: Best-of-N Chain Sampling
# ---------------------------------------------------------------------------

def run_config3_best_of_n(
    question: str, vs: FAISS,
    llm: ChatOllama, config: CoRAGConfig,
) -> CoRAGTrace:
    """
    Configuration 3: CoRAG Best-of-N Chain Sampling.

    Samples N independent retrieval chains with temperature=CHAIN_TEMPERATURE.
    Selects the best chain by the highest quality score (proxy for lowest
    failure penalty). Generates the final answer from the selected best chain.

    This is the test-time compute scaling strategy from Wang et al. (2025):
        At N=4: ~4x the LLM cost of greedy but higher quality because:
        (a) The best chain selects the decomposition that best bridges the hops.
        (b) Stochastic sub-query generation explores diverse query phrasings,
            some of which may retrieve better documents than the greedy path.

    Best-of-N is particularly effective when retrieval recall is the bottleneck:
    if the correct document IS in the corpus but the greedy sub-query misses it,
    a different random sub-query formulation might phrase it in a way that hits.

    Selection criterion: highest score_chain_quality() (Azure proxy for
    log P(final_answer | Q, chain) which is the original paper's metric).

    LLM calls: (2 * L + 1) * N + 1_quality_per_chain + 1_final_answer
               For N=4, L=4: up to ~38 LLM calls.
    """
    trace = CoRAGTrace(question=question, strategy="Config3_CoRAG_BestOfN")
    t0    = time.perf_counter()

    chains: List[RetrievalChain] = []
    total_ret_ms  = 0.0
    total_gen_ms  = 0.0
    total_llm     = 0

    logger.info("Config3: Sampling %d chains (temperature=%.1f) ...",
                config.BEST_OF_N, config.CHAIN_TEMPERATURE)

    for n in range(config.BEST_OF_N):
        logger.info("  Sampling chain %d/%d ...", n + 1, config.BEST_OF_N)
        c = _build_greedy_chain(
            question, vs, llm, config, temperature=config.CHAIN_TEMPERATURE
        )
        # Score the chain quality (proxy penalty)
        t_score = time.perf_counter()
        c.quality_score = score_chain_quality(question, c, llm, config)
        total_gen_ms += (time.perf_counter() - t_score) * 1000

        chains.append(c)
        total_ret_ms += sum(s.retrieval_ms for s in c.steps)
        total_gen_ms += sum(s.generation_ms for s in c.steps)
        total_llm    += c.total_llm_calls + 1  # +1 for quality scoring

        logger.info(
            "  Chain %d: steps=%d, failures=%d, quality=%.2f",
            n + 1, c.n_steps(), c.n_failures(), c.quality_score,
        )

    # Select chain with highest quality score (lowest failure probability)
    best_chain = max(chains, key=lambda c: c.quality_score)
    logger.info(
        "Config3: Selected chain %d (quality=%.2f, steps=%d)",
        chains.index(best_chain) + 1, best_chain.quality_score, best_chain.n_steps(),
    )

    t_gen = time.perf_counter()
    best_chain.final_answer = generate_final_answer(question, best_chain, llm, config)
    total_gen_ms += (time.perf_counter() - t_gen) * 1000
    total_llm    += 1

    trace.chains         = chains
    trace.selected_chain = best_chain
    trace.final_answer   = best_chain.final_answer
    trace.retrieval_ms   = total_ret_ms
    trace.generation_ms  = total_gen_ms
    trace.llm_calls      = total_llm
    trace.total_ms       = (time.perf_counter() - t0) * 1000
    return trace


In [12]:

 # ---------------------------------------------------------------------------
# CONFIG 4: BFS Tree Search
# ---------------------------------------------------------------------------

def run_config4_bfs_tree_search(
    question: str, vs: FAISS,
    llm: ChatOllama, config: CoRAGConfig,
) -> CoRAGTrace:
    """
    Configuration 4: CoRAG BFS Tree Search.

    At each step, expands BFS_BEAM_WIDTH candidate sub-queries from the
    current state. For each candidate, runs BFS_ROLLOUT_COUNT greedy rollouts
    (completions of the chain from that state). Selects the candidate with
    the lowest average penalty (highest average quality) for the next step.

    Tree search explores the reasoning space more exhaustively than Best-of-N:
        Best-of-N:   samples N independent chains from the START.
        Tree search: at each STEP, selects the locally best continuation.
                     This is BFS with rollout-based lookahead.

    From Wang et al. (2025): tree search achieves the best performance on the
    hardest queries (MuSiQue, Bamboogle) at the cost of highest compute. The
    Pareto frontier improvement from Best-of-N to Tree Search follows the same
    log-linear scaling law as the other strategies.

    BFS variant: at each of the L steps, generate BFS_BEAM_WIDTH sub-queries
    (with temperature=CHAIN_TEMPERATURE), evaluate each by BFS_ROLLOUT_COUNT
    greedy rollouts, select the best, advance the state, repeat.

    LLM calls: per step: BFS_BEAM_WIDTH * (2 * L_remaining + 1) * BFS_ROLLOUT_COUNT
               For BFS_BEAM_WIDTH=2, BFS_ROLLOUT_COUNT=2, L=4:
               Approximately 30-60 LLM calls total.
    """
    trace = CoRAGTrace(question=question, strategy="Config4_CoRAG_BFS_TreeSearch")
    t0    = time.perf_counter()

    current_chain = RetrievalChain(question=question)
    total_ret_ms  = 0.0
    total_gen_ms  = 0.0
    total_llm     = 0
    best_score_overall = 0.0
    last_selected_score = 0.0

    for step_num in range(1, config.CHAIN_LENGTH + 1):
        # --- Generate BFS_BEAM_WIDTH candidate sub-queries ----------------
        logger.info(
            "Config4 BFS step %d: generating %d candidate sub-queries ...",
            step_num, config.BFS_BEAM_WIDTH,
        )
        candidate_subqueries: List[str] = []
        for _ in range(config.BFS_BEAM_WIDTH):
            sq = generate_next_subquery(
                question, current_chain, llm, config,
                temperature=config.CHAIN_TEMPERATURE,
            )
            total_llm += 1
            if "DONE" in sq.upper() and len(sq) < 10:
                # Guard against occasional premature DONE before any retrieval step.
                if step_num == 1 and not current_chain.steps:
                    logger.info(
                        "  BFS step %d: ignoring premature DONE at chain start",
                        step_num,
                    )
                    continue
                logger.info("  BFS step %d: LLM signaled DONE", step_num)
                break
            if sq and sq not in candidate_subqueries:
                candidate_subqueries.append(sq[:200])

        if not candidate_subqueries:
            if step_num == 1 and not current_chain.steps:
                fallback_sq = question[:200]
                candidate_subqueries.append(fallback_sq)
                logger.info(
                    "  BFS step %d: using original question as fallback sub-query",
                    step_num,
                )
            else:
                break

        # --- For each candidate: run rollouts + compute average quality ---
        best_sq    = candidate_subqueries[0]
        best_score = -1.0
        best_step_data: Optional[Tuple[RetrievalStep, float]] = None

        for sq in candidate_subqueries:
            rollout_scores: List[float] = []

            docs, ret_ms = retrieve_for_subquery(sq, vs, config.TOP_K_RETRIEVAL)
            total_ret_ms += ret_ms

            t_sa = time.perf_counter()
            sub_ans, is_fail, is_terminal = generate_sub_answer(sq, docs, llm, config)
            total_gen_ms += (time.perf_counter() - t_sa) * 1000
            total_llm += 1

            # Create a prospective chain with this step
            prospective_chain = deepcopy(current_chain)
            step_obj = RetrievalStep(
                step_num=step_num, sub_query=sq, retrieved=docs,
                sub_answer=sub_ans, is_failure=is_fail, is_terminal=is_terminal,
                retrieval_ms=ret_ms,
            )
            prospective_chain.steps.append(step_obj)

            # Rollouts: complete the chain from this state
            for _ in range(config.BFS_ROLLOUT_COUNT):
                rollout_chain = _build_greedy_chain(
                    question, vs, llm, config,
                    max_steps=config.CHAIN_LENGTH - step_num,
                    temperature=0.0,
                )
                # Merge prospective + rollout steps for quality scoring
                merged_chain = deepcopy(prospective_chain)
                merged_chain.steps.extend(rollout_chain.steps)
                total_llm += rollout_chain.total_llm_calls

                t_score = time.perf_counter()
                score = score_chain_quality(question, merged_chain, llm, config)
                total_gen_ms += (time.perf_counter() - t_score) * 1000
                total_llm += 1
                rollout_scores.append(score)

            avg_score = sum(rollout_scores) / max(len(rollout_scores), 1)
            logger.info(
                "  Candidate '%s ...': avg_quality=%.2f",
                sq[:40], avg_score,
            )

            if avg_score > best_score:
                best_score  = avg_score
                best_sq     = sq
                best_step_data = (step_obj, avg_score)

        # Advance state with best candidate
        if best_step_data is not None:
            best_step, _ = best_step_data
            current_chain.steps.append(best_step)
            best_score_overall = best_score
            last_selected_score = best_score
            logger.info(
                "Config4 BFS step %d: selected '%s ...' (quality=%.2f)",
                step_num, best_sq[:40], best_score,
            )
            if best_step.is_terminal and step_num >= 2:
                break
        else:
            break

    t_gen = time.perf_counter()
    current_chain.final_answer = generate_final_answer(question, current_chain, llm, config)
    total_gen_ms += (time.perf_counter() - t_gen) * 1000
    total_llm    += 1

    current_chain.quality_score = last_selected_score
    trace.selected_chain = current_chain
    trace.chains         = [current_chain]
    trace.final_answer   = current_chain.final_answer
    trace.retrieval_ms   = total_ret_ms
    trace.generation_ms  = total_gen_ms
    trace.llm_calls      = total_llm
    trace.total_ms       = (time.perf_counter() - t0) * 1000
    return trace


In [13]:

# ---------------------------------------------------------------------------
# CONFIG 5: CoRAG Iterative Self-Correction [BEST inference-time strategy]
# ---------------------------------------------------------------------------

def run_config5_iterative_self_correct(
    question: str, vs: FAISS,
    llm: ChatOllama, config: CoRAGConfig,
) -> CoRAGTrace:
    """
    Configuration 5: CoRAG Iterative Self-Correction [BEST INFERENCE STRATEGY].

    This configuration combines greedy chain building with explicit self-
    correction on retrieval failure. When the retriever returns no useful
    documents for a sub-query (failure sentinel detected), instead of advancing
    to the next reasoning step, the model REFORMULATES the failed query and
    retries retrieval.

    This implements the behavior described in Wang et al. (2025):
        "CoRAG self-corrects by verifying the poet's name and country of origin
        through additional retrieval steps."
        "CoRAG dynamically conducts query reformulation when the retrieved
        information proves unhelpful."

    Self-correction mechanism:
        At each step:
            1. Generate next sub-query Q_t.
            2. Retrieve and generate sub-answer A_t.
            3. IF A_t is a failure sentinel:
               a. Reformulate Q_t -> Q_t' using context.
               b. Retry retrieval with Q_t'.
               c. Generate new sub-answer A_t'.
               d. Repeat up to MAX_REFORMULATION_ATTEMPTS times.
            4. IF sub-answer is still failure after all attempts: log and continue.
            5. ELSE: advance chain with successful sub-answer.

    This self-correction transforms temporary retrieval failures into corrected
    steps within the SAME chain, rather than requiring Best-of-N to find a
    non-failing chain among N samples. It is more compute-efficient than
    Best-of-N for queries where retrieval failure is a phrasing issue
    (synonyms, paraphrases) rather than a data coverage issue.

    From a production standpoint this is the recommended inference strategy:
        - More adaptive than greedy (handles retrieval failures actively).
        - Less expensive than Best-of-N (one chain + targeted reformulations).
        - More interpretable than tree search (linear audit trail).

    LLM calls: 2-3 per step (sub-query + sub-answer + optional reformulation)
               + 1 for final answer. For L=4: typically 10-15 LLM calls.
    """
    trace = CoRAGTrace(question=question,
                       strategy="Config5_CoRAG_IterativeSelfCorrect [BEST]")
    t0 = time.perf_counter()

    chain = RetrievalChain(question=question)
    total_llm_calls = 0
    total_ret_ms    = 0.0
    total_gen_ms    = 0.0

    for step_num in range(1, config.CHAIN_LENGTH + 1):
        # --- Generate next sub-query (1 LLM call) -------------------------
        next_q = generate_next_subquery(question, chain, llm, config, temperature=0.0)
        total_llm_calls += 1

        if "DONE" in next_q.upper() and len(next_q) < 10:
            # Guard against premature termination before any retrieval step.
            if step_num == 1 and not chain.steps:
                logger.info(
                    "  Step %d: premature DONE at chain start; using original question",
                    step_num,
                )
                next_q = question
            else:
                logger.info("  Step %d: LLM signaled DONE", step_num)
                break

        next_q = next_q[:200]

        # --- Retrieve + generate sub-answer + self-correction loop --------
        reformulation_attempts = 0
        current_q = next_q
        step_is_reformulation = False

        while True:
            docs, ret_ms = retrieve_for_subquery(current_q, vs, config.TOP_K_RETRIEVAL)
            total_ret_ms += ret_ms

            t_sa = time.perf_counter()
            sub_ans, is_fail, is_terminal = generate_sub_answer(
                current_q, docs, llm, config
            )
            total_gen_ms  += (time.perf_counter() - t_sa) * 1000
            total_llm_calls += 1

            if not is_fail:
                break

            if reformulation_attempts >= config.MAX_REFORMULATION_ATTEMPTS:
                logger.info(
                    "  Step %d: max reformulations reached for '%s ...'",
                    step_num, current_q[:40],
                )
                break

            # Reformulate and retry
            logger.info(
                "  Step %d: failure detected for '%s ...' -> reformulating ...",
                step_num, current_q[:40],
            )
            t_ref = time.perf_counter()
            reformulated_q = generate_reformulated_subquery(current_q, chain, llm, config)
            total_gen_ms    += (time.perf_counter() - t_ref) * 1000
            total_llm_calls += 1

            reformulation_attempts += 1
            current_q = reformulated_q[:200]
            step_is_reformulation = True
            logger.info(
                "  Step %d: reformulated to '%s ...'",
                step_num, current_q[:40],
            )

        step = RetrievalStep(
            step_num=step_num, sub_query=current_q, retrieved=docs,
            sub_answer=sub_ans, is_failure=is_fail, is_terminal=is_terminal,
            retrieval_ms=ret_ms, reformulated=step_is_reformulation,
        )
        chain.steps.append(step)

        logger.info(
            "  Step %d [reform=%s, fail=%s]: '%s' -> '%s ...'",
            step_num, step_is_reformulation, is_fail,
            current_q[:35], sub_ans[:40],
        )

        if is_terminal and step_num >= 2:
            logger.info("  Step %d: terminal answer. Stopping chain.", step_num)
            break

    # Final answer synthesis
    t_gen = time.perf_counter()
    final_answer = generate_final_answer(question, chain, llm, config)
    total_gen_ms    += (time.perf_counter() - t_gen) * 1000
    total_llm_calls += 1

    chain.final_answer    = final_answer
    chain.total_llm_calls = total_llm_calls

    trace.selected_chain = chain
    trace.chains         = [chain]
    trace.final_answer   = final_answer
    trace.retrieval_ms   = total_ret_ms
    trace.generation_ms  = total_gen_ms
    trace.llm_calls      = total_llm_calls
    trace.total_ms       = (time.perf_counter() - t0) * 1000
    return trace


In [14]:

# ===========================================================================
# SECTION 6 -- COMPARATIVE RUNNER
# ===========================================================================

def run_all_configs(
    question: str, vs: FAISS,
    llm: ChatOllama, config: CoRAGConfig,
) -> Dict[str, CoRAGTrace]:
    print("\n" + "#" * 78)
    print(f"QUERY: {question}")
    print("#" * 78)

    runners = {
        "Config1_Baseline_SingleStep":          lambda q: run_config1_baseline(
            q, vs, llm, config),
        "Config2_CoRAG_GreedyChain":            lambda q: run_config2_greedy_chain(
            q, vs, llm, config),
        "Config3_CoRAG_BestOfN":               lambda q: run_config3_best_of_n(
            q, vs, llm, config),
        "Config4_CoRAG_BFS_TreeSearch":         lambda q: run_config4_bfs_tree_search(
            q, vs, llm, config),
        "Config5_CoRAG_IterativeSelfCorrect":   lambda q: run_config5_iterative_self_correct(
            q, vs, llm, config),
    }

    traces: Dict[str, CoRAGTrace] = {}
    for label, fn in runners.items():
        print(f"\n{'='*62}\nRUNNING: {label}\n{'='*62}")
        try:
            trace = fn(question)
            trace.print_trace()
            traces[label] = trace
        except Exception as exc:
            logger.error("Config %s failed: %s", label, exc, exc_info=True)

    print("\n" + "=" * 78)
    print("COG-RETRIEVAL COMPARISON SUMMARY")
    print(f"{'Config':<47} {'Steps':>5} {'Fails':>5} {'LLM':>5} "
          f"{'Ret_ms':>7} {'Gen_ms':>7} {'Total_ms':>9}")
    print("-" * 78)
    for lbl, tr in traces.items():
        sc = tr.selected_chain
        steps = sc.n_steps() if sc else 0
        fails = sc.n_failures() if sc else 0
        print(f"{lbl:<47} {steps:>5} {fails:>5} {tr.llm_calls:>5} "
              f"{tr.retrieval_ms:>7.0f} {tr.generation_ms:>7.0f} {tr.total_ms:>9.0f}")
    print("=" * 78 + "\n")
    return traces



In [15]:
 """
    End-to-end CoRAG pipeline orchestrator.

    INDEXING (run once, cached):
        Load 3 arXiv PDFs -> chunk -> BGE-large embeddings -> FAISS index.
        No LLM calls during indexing. Fast (<2 min).

    QUERY (per query):
        Config 1: 1 LLM call (baseline).
        Config 2: 2*L + 1 LLM calls (greedy chain). Typical: ~9.
        Config 3: N * (2*L + 2) LLM calls (best-of-N). Typical: ~40.
        Config 4: BFS_BEAM_WIDTH * BFS_ROLLOUT_COUNT * (2*L + 1) LLM calls.
                  Typical: ~50. Highest compute, highest quality.
        Config 5: 2-3 per step + 1 final. Typical: ~12. Best cost/quality.

    PRODUCTION SCALING NOTE:
        For the full fine-tuned CoRAG system (Wang et al., 2025):
            1. Use rejection sampling to generate chains from your QA dataset.
            2. Fine-tune Llama-3.1-8B on the augmented chains.
            3. The fine-tuned model natively knows WHEN to generate sub-queries
               vs WHEN to write "DONE" -- removing the need for config-level
               prompting for termination.
            4. Use E5-large or E5-Mistral as the retriever.
            5. Scale test-time compute with Best-of-N or Tree Search.
        This pipeline implements the inference-time behaviors using GPT-4o,
        which approximates the fine-tuned model's capabilities via prompting.
    """

'\n   End-to-end CoRAG pipeline orchestrator.\n\n   INDEXING (run once, cached):\n       Load 3 arXiv PDFs -> chunk -> BGE-large embeddings -> FAISS index.\n       No LLM calls during indexing. Fast (<2 min).\n\n   QUERY (per query):\n       Config 1: 1 LLM call (baseline).\n       Config 2: 2*L + 1 LLM calls (greedy chain). Typical: ~9.\n       Config 3: N * (2*L + 2) LLM calls (best-of-N). Typical: ~40.\n       Config 4: BFS_BEAM_WIDTH * BFS_ROLLOUT_COUNT * (2*L + 1) LLM calls.\n                 Typical: ~50. Highest compute, highest quality.\n       Config 5: 2-3 per step + 1 final. Typical: ~12. Best cost/quality.\n\n   PRODUCTION SCALING NOTE:\n       For the full fine-tuned CoRAG system (Wang et al., 2025):\n           1. Use rejection sampling to generate chains from your QA dataset.\n           2. Fine-tune Llama-3.1-8B on the augmented chains.\n           3. The fine-tuned model natively knows WHEN to generate sub-queries\n              vs WHEN to write "DONE" -- removing the 

In [16]:
config = CoRAGConfig()
logger.info("=== CoRAG Chain-of-Retrieval Pipeline Starting ===")


2026-05-25 22:10:15 | INFO     | corag | === CoRAG Chain-of-Retrieval Pipeline Starting ===


In [17]:
pdf_paths  = download_pdfs(config)


2026-05-25 22:10:15 | INFO     | corag | Cached: attention_is_all_you_need.pdf
2026-05-25 22:10:15 | INFO     | corag | Cached: bert_pretraining_transformers.pdf
2026-05-25 22:10:15 | INFO     | corag | Cached: rag_knowledge_intensive_nlp.pdf


In [18]:
chunks     = load_and_chunk(pdf_paths, config)


2026-05-25 22:10:17 | INFO     | corag |   attention_is_all_you_need.pdf -> 104 chunks
2026-05-25 22:10:19 | INFO     | corag |   bert_pretraining_transformers.pdf -> 173 chunks
2026-05-25 22:10:20 | INFO     | corag |   rag_knowledge_intensive_nlp.pdf -> 181 chunks
2026-05-25 22:10:20 | INFO     | corag | Total chunks: 458


In [19]:
embeddings = build_bge_embeddings(config)


2026-05-25 22:10:20 | INFO     | corag | Loading BGE: BAAI/bge-large-en-v1.5
2026-05-25 22:10:20 | INFO     | sentence_transformers.SentenceTransformer | Load pretrained SentenceTransformer: BAAI/bge-large-en-v1.5


C:\Users\dhanu\AppData\Local\Temp\ipykernel_3900\1427019833.py:52: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  return HuggingFaceEmbeddings(


2026-05-25 22:10:22 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-05-25 22:10:22 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/modules.json "HTTP/1.1 200 OK"
2026-05-25 22:10:22 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"


2026-05-25 22:10:22 | WARNING  | huggingface_hub.utils._http | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-05-25 22:10:22 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-05-25 22:10:22 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-05-25 22:10:23 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-05-25 22:10:23 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 4447.95it/s]
BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


2026-05-25 22:10:25 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-25 22:10:25 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/config.json "HTTP/1.1 200 OK"
2026-05-25 22:10:26 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-25 22:10:26 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/tokenizer_config.json "HTTP/1.1 200 OK"
2026-05-25 22:10:26 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-large-en-v1.5/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-05-25 22:10:26 | INFO     | httpx |

In [20]:
vs         = build_faiss(chunks, embeddings, config)


2026-05-25 22:10:27 | INFO     | faiss.loader | Loading faiss with AVX512 support.
2026-05-25 22:10:27 | INFO     | faiss.loader | Could not load library with AVX512 support due to:
ModuleNotFoundError("No module named 'faiss.swigfaiss_avx512'")
2026-05-25 22:10:27 | INFO     | faiss.loader | Loading faiss with AVX2 support.
2026-05-25 22:10:28 | INFO     | faiss.loader | Successfully loaded faiss with AVX2 support.
2026-05-25 22:10:28 | INFO     | corag | FAISS loaded. Vectors: 458


In [21]:
llm        = build_ollama_llm(config, temperature=config.ANSWER_TEMPERATURE)


In [22]:
# Multi-hop questions designed to stress-test single-step retrieval
demo_questions = [
    # 2-hop: find paper -> find specific detail in that paper
    "What pre-training objective does the model that uses masked language modeling use, and how many transformer encoder layers does it use in the base configuration?",

    # 2-hop: find concept -> find specification of that concept
    "What is the dimensionality of the query and key vectors in the base Transformer model, and how many attention heads are used?",

    # 3-hop: connect information across three papers
    "How does the retrieval mechanism in the RAG paper differ from the attention mechanism in the Transformer paper in terms of how external information is incorporated?",

    # self-correction test: deliberately ambiguous phrasing
    "What is the size of the model described in the paper that introduced the architecture used for pre-training on large text corpora with bidirectional context?",

    # Requires bridging encoder/decoder architecture details
    "What are the architectural differences between the encoder and decoder blocks in the Transformer, specifically regarding which attention sub-layers are present in each?",
]

In [23]:
for question in demo_questions:
    run_all_configs(question, vs, llm, config)

logger.info("=== CoRAG Pipeline Demo Complete ===")



##############################################################################
QUERY: What pre-training objective does the model that uses masked language modeling use, and how many transformer encoder layers does it use in the base configuration?
##############################################################################

RUNNING: Config1_Baseline_SingleStep
2026-05-25 22:10:38 | INFO     | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"

TRACE: Config1_Baseline_SingleStep
Query: What pre-training objective does the model that uses masked language model
  Chain steps:   0 | Retrieval failures: 0 | Quality score: 0.00

  CHAIN HISTORY:

  LLM calls: 1
  Latency: retrieval=2458ms  gen=6712ms  total=9171ms

  FINAL ANSWER:
  The provided documents do not contain sufficient information.


RUNNING: Config2_CoRAG_GreedyChain
2026-05-25 22:10:44 | INFO     | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-05-25 22:10:44 | INF

In [ ]:
# Debug Config1 directly to capture hidden runner exception
try:
    _cfg1 = run_config1_baseline(test_question, vs, llm, config)
    print("Config1 direct run succeeded")
    print(_cfg1.final_answer)
except Exception as e:
    import traceback
    print("Config1 direct run failed:")
    traceback.print_exc()

2026-05-25 21:34:44 | INFO     | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
Config1 direct run succeeded
The provided documents do not contain sufficient information.


In [ ]:
# Quick regression test for Config 4 after bug fix
q = demo_questions[0]
trace4 = run_config4_bfs_tree_search(q, vs, llm, config)
print('Config4 status: OK')
print('Steps:', trace4.selected_chain.n_steps() if trace4.selected_chain else 0)
print('LLM calls:', trace4.llm_calls)
print('Answer:', trace4.final_answer)

2026-05-25 21:34:50 | INFO     | corag | Config4 BFS step 1: generating 2 candidate sub-queries ...
2026-05-25 21:34:54 | INFO     | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-05-25 21:34:55 | INFO     | corag |   BFS step 1: LLM signaled DONE
2026-05-25 21:34:57 | INFO     | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
Config4 status: OK
Steps: 0
LLM calls: 2
Answer: The provided documents do not contain sufficient information to answer this question.
